# Classificazione binaria



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Costruirai un classificatore binario per prevedere le cancellazioni di prenotazioni alberghiere.


In [ ]:
# Configura i grafici.
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
# Imposta i valori predefiniti di Matplotlib.
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('animation', html='html5')

# Configura il sistema di verifica degli esercizi.
from learntools.core import binder
binder.bind(globals())
from learntools.deep_learning_intro.ex6 import *

Carica il dataset **Hotel Cancellations**.


In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

hotel = pd.read_csv('../input/dl-course-data/hotel.csv')

X = hotel.copy()
y = X.pop('is_canceled')

X['arrival_date_month'] = \
    X['arrival_date_month'].map(
        {'January':1, 'February': 2, 'March':3,
         'April':4, 'May':5, 'June':6, 'July':7,
         'August':8, 'September':9, 'October':10,
         'November':11, 'December':12}
    )

features_num = [
    "lead_time", "arrival_date_week_number",
    "arrival_date_day_of_month", "stays_in_weekend_nights",
    "stays_in_week_nights", "adults", "children", "babies",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "required_car_parking_spaces",
    "total_of_special_requests", "adr",
]
features_cat = [
    "hotel", "arrival_date_month", "meal",
    "market_segment", "distribution_channel",
    "reserved_room_type", "deposit_type", "customer_type",
]

transformer_num = make_pipeline(
    SimpleImputer(strategy="constant"), # Sono presenti alcuni valori mancanti.
    StandardScaler(),
)
transformer_cat = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="NA"),
    OneHotEncoder(handle_unknown='ignore'),
)

preprocessor = make_column_transformer(
    (transformer_num, features_num),
    (transformer_cat, features_cat),
)

# stratify mantiene proporzioni simili delle classi nei diversi sottoinsiemi.
X_train, X_valid, y_train, y_valid = \
    train_test_split(X, y, stratify=y, train_size=0.75)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)

input_shape = [X_train.shape[1]]

# 1. Definire il modello
Il modello usa batch normalization e dropout. Il disegno è diviso in blocchi per leggibilità; puoi definire i layer uno dopo l'altro.

![Architettura: BatchNorm, Dense, BatchNorm, Dropout, Dense, BatchNorm, Dropout, Dense](https://storage.googleapis.com/kaggle-media/learn/images/V04o59Z.png)

Riproduci l'architettura indicata.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# SCRIVI QUI IL CODICE: definisci il modello rappresentato nel diagramma.
model = ____

# Controlla la risposta.
q_1.check()

# 2. Optimizer, loss e metrica
Compila il modello con **Adam**, **binary cross-entropy** e **binary accuracy**.


In [ ]:
# SCRIVI QUI IL CODICE
____

# Controlla la risposta.
q_2.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_2.hint()
#q_2.solution()

Esegui il training e osserva le curve. Nell'ambiente originale potevano servire circa 60–70 epoche e uno o due minuti; la durata dipende dall'ambiente.


In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=5,
    min_delta=0.001,
    restore_best_weights=True,
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=200,
    callbacks=[early_stopping],
)

history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot(title="Cross-entropy")
history_df.loc[:, ['binary_accuracy', 'val_binary_accuracy']].plot(title="Accuracy")

# 3. Addestrare e valutare
Le curve indicano underfitting oppure overfitting? La riduzione della cross-entropy è accompagnata da un miglioramento dell'accuracy?


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_3.check()

# Conclusione
Hai completato il percorso introduttivo di Kaggle sul deep learning. Puoi approfondire computer vision e classificazione del sentiment.

Il testo originale suggerisce queste competizioni introduttive:

* [Petals to the Metal](https://www.kaggle.com/c/tpu-getting-started): immagini con TPU.
* [I'm Something of a Painter Myself](https://www.kaggle.com/c/gan-getting-started): immagini con GAN.
* [Real or Not? NLP with Disaster Tweets](https://www.kaggle.com/c/nlp-getting-started): classificazione di tweet.
* [Contradictory, My Dear Watson](https://www.kaggle.com/c/contradictory-my-dear-watson): relazioni fra frasi.

In AI Lab puoi invece proseguire con un progetto scolastico, dimensionato per risorse gratuite.
